# Práctica 3 - Tarea 1: Detección y conteo de monedas
## Grupo 33

El objetivo de esta tarea es:
1. Detectar las monedas en la imagen estática proporcionada (`Monedas.jpg`) y determinar la cantidad de dinero presente.
2. Diseñar un demostrador en vivo con la webcam que determine el dinero en la escena.

**No se utilizan modelos de aprendizaje profundo.**

### Estrategia
- **Detección**: Transformada de Hough para círculos.
- **Calibración**: Se identifica una moneda de referencia (la más grande = 2€, diámetro = 25.75 mm) para obtener la relación píxeles/mm. También se permite calibración interactiva haciendo clic.
- **Clasificación**: A partir del diámetro en mm y el color (HSV) se clasifica cada moneda.
- **Color**: Se distingue entre monedas cobrizo/cobre (1c, 2c, 5c), doradas/Nordic gold (10c, 20c, 50c) y bimetálicas (1€, 2€).


In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt


## Diámetros oficiales de monedas Euro (mm)

| Moneda | Diámetro (mm) | Valor (€) | Color |
|--------|--------------|-----------|-------|
| 2 euro | 25.75 | 2.00 | Bimetálica (plata exterior, dorado interior) |
| 1 euro | 23.25 | 1.00 | Bimetálica (dorado exterior, plata interior) |
| 50 céntimos | 24.25 | 0.50 | Dorada (Nordic gold) |
| 20 céntimos | 22.25 | 0.20 | Dorada (Nordic gold) |
| 10 céntimos | 19.75 | 0.10 | Dorada (Nordic gold) |
| 5 céntimos | 21.25 | 0.05 | Cobriza |
| 2 céntimos | 18.75 | 0.02 | Cobriza |
| 1 céntimo | 16.25 | 0.01 | Cobriza |


In [ ]:
# Definición de monedas Euro: (nombre, diámetro_mm, valor_euros, tipo_color)
# tipo_color: "bimetalica", "dorada", "cobriza"
MONEDAS_EURO = [
    ("2 euro",      25.75, 2.00, "bimetalica"),
    ("50 centimos", 24.25, 0.50, "dorada"),
    ("1 euro",      23.25, 1.00, "bimetalica"),
    ("20 centimos", 22.25, 0.20, "dorada"),
    ("5 centimos",  21.25, 0.05, "cobriza"),
    ("10 centimos", 19.75, 0.10, "dorada"),
    ("2 centimos",  18.75, 0.02, "cobriza"),
    ("1 centimo",   16.25, 0.01, "cobriza"),
]

# Colores para dibujar según tipo
COLORES_TIPO = {
    "bimetalica": (0, 255, 255),   # Amarillo (BGR)
    "dorada":     (0, 200, 255),    # Naranja (BGR)
    "cobriza":    (0, 100, 200),    # Marrón/cobre (BGR)
    "desconocida": (255, 255, 255), # Blanco
}


In [ ]:
def clasificar_color_moneda(img_bgr, cx, cy, radio):
    """
    Clasifica el color de una moneda analizando la región central
    en el espacio HSV.
    
    Retorna: "bimetalica", "dorada" o "cobriza"
    """
    hsv = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)
    h, w = img_bgr.shape[:2]
    
    # Crear máscara circular en la zona central (50% del radio)
    mask = np.zeros((h, w), dtype=np.uint8)
    cv2.circle(mask, (int(cx), int(cy)), max(1, int(radio * 0.5)), 255, -1)
    
    # Valores medios HSV en la zona central
    mean_hsv = cv2.mean(hsv, mask=mask)[:3]
    hue, sat, val = mean_hsv
    
    # Analizar la zona anular exterior (entre 60% y 90% del radio)
    mask_outer = np.zeros((h, w), dtype=np.uint8)
    cv2.circle(mask_outer, (int(cx), int(cy)), max(1, int(radio * 0.9)), 255, -1)
    mask_inner = np.zeros((h, w), dtype=np.uint8)
    cv2.circle(mask_inner, (int(cx), int(cy)), max(1, int(radio * 0.6)), 255, -1)
    mask_ring = cv2.subtract(mask_outer, mask_inner)
    mean_hsv_outer = cv2.mean(hsv, mask=mask_ring)[:3]
    
    # Diferencia entre centro y borde
    sat_diff = abs(mean_hsv[1] - mean_hsv_outer[1])
    hue_diff = abs(mean_hsv[0] - mean_hsv_outer[0])
    
    # Bimetálica: gran diferencia de saturación o hue entre centro y borde
    if sat_diff > 30 or hue_diff > 10:
        return "bimetalica"
    
    # Cobriza: hue bajo (rojizo) y alta saturación
    if hue < 12 and sat > 100:
        return "cobriza"
    
    # Dorada: hue entre 12-30 y saturación moderada-alta
    if 12 <= hue <= 30 and sat > 80:
        return "dorada"
    
    # Por defecto, si baja saturación probablemente bimetálica
    if sat < 40:
        return "bimetalica"
    
    return "dorada"


In [ ]:
def clasificar_moneda(diametro_mm, tipo_color):
    """
    Clasifica una moneda a partir de su diámetro en mm y su tipo de color.
    Busca la moneda más cercana en diámetro compatible con el color.
    
    Retorna: (nombre, valor_euros)
    """
    mejor_match = None
    mejor_dist = float("inf")
    
    for nombre, diam, valor, color in MONEDAS_EURO:
        dist = abs(diametro_mm - diam)
        
        # Priorizar monedas del mismo tipo de color
        if color == tipo_color:
            dist_efectiva = dist
        else:
            # Penalización por no coincidir el color
            dist_efectiva = dist + 5.0
        
        if dist_efectiva < mejor_dist:
            mejor_dist = dist_efectiva
            mejor_match = (nombre, valor)
    
    return mejor_match


In [ ]:
def detectar_y_contar_monedas(img_bgr, px_por_mm, 
                              min_radius_px=15, max_radius_px=200,
                              param1=100, param2=50, min_dist=50,
                              blur_ksize=7):
    """
    Detecta monedas en una imagen usando la Transformada de Hough para círculos.
    Clasifica cada moneda por tamaño y color.
    
    Args:
        img_bgr: Imagen en BGR
        px_por_mm: Relación píxeles por milímetro
        min_radius_px, max_radius_px: Rango de radios para detección
        param1, param2: Parámetros de HoughCircles
        min_dist: Distancia mínima entre centros
        blur_ksize: Tamaño del kernel de suavizado
    
    Retorna: (img_resultado, lista_monedas, total_euros)
    """
    img_result = img_bgr.copy()
    gris = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    gris_blur = cv2.medianBlur(gris, blur_ksize)
    
    # Detectar círculos con Hough
    circulos = cv2.HoughCircles(
        gris_blur,
        cv2.HOUGH_GRADIENT,
        dp=1,
        minDist=min_dist,
        param1=param1,
        param2=param2,
        minRadius=min_radius_px,
        maxRadius=max_radius_px,
    )
    
    monedas_detectadas = []
    total = 0.0
    
    if circulos is not None:
        circulos = np.uint16(np.around(circulos))
        for circ in circulos[0, :]:
            cx, cy, radio_px = circ
            
            # Convertir radio a diámetro en mm
            diametro_mm = (2 * radio_px) / px_por_mm
            
            # Clasificar color
            tipo_color = clasificar_color_moneda(img_bgr, cx, cy, radio_px)
            
            # Clasificar moneda
            nombre, valor = clasificar_moneda(diametro_mm, tipo_color)
            total += valor
            
            monedas_detectadas.append({
                "nombre": nombre,
                "valor": valor,
                "centro": (int(cx), int(cy)),
                "radio_px": int(radio_px),
                "diametro_mm": diametro_mm,
                "tipo_color": tipo_color,
            })
            
            # Dibujar círculo y etiqueta
            color_dibujo = COLORES_TIPO.get(tipo_color, (255, 255, 255))
            cv2.circle(img_result, (int(cx), int(cy)), int(radio_px), color_dibujo, 2)
            cv2.circle(img_result, (int(cx), int(cy)), 3, color_dibujo, -1)
            
            # Texto con nombre y valor
            texto = f"{nombre} ({valor:.2f}E)"
            cv2.putText(img_result, texto,
                       (int(cx) - int(radio_px), int(cy) - int(radio_px) - 10),
                       cv2.FONT_HERSHEY_SIMPLEX, 0.45, color_dibujo, 1)
            
            # Diámetro en mm
            texto_mm = f"{diametro_mm:.1f}mm"
            cv2.putText(img_result, texto_mm,
                       (int(cx) - 20, int(cy) + 5),
                       cv2.FONT_HERSHEY_SIMPLEX, 0.35, (255, 255, 255), 1)
    
    # Mostrar total en la imagen
    texto_total = f"Total: {total:.2f} EUR ({len(monedas_detectadas)} monedas)"
    cv2.putText(img_result, texto_total, (10, 30),
               cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)
    
    return img_result, monedas_detectadas, total


## Paso 1: Detección en la imagen estática (`Monedas.jpg`)

Cargamos la imagen y detectamos los círculos. Usamos la moneda más grande (2€) como referencia
para calibrar la relación píxeles/mm.


In [ ]:
# Cargar imagen
img = cv2.imread("Monedas.jpg")
if img is None:
    raise FileNotFoundError("No se encontró Monedas.jpg")
print(f"Imagen cargada: {img.shape}")

# Mostrar la imagen original
img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
plt.figure(figsize=(6, 12))
plt.imshow(img_rgb)
plt.title("Imagen original - Monedas.jpg")
plt.axis("off")
plt.show()


In [ ]:
# Paso 1: Detectar círculos con Hough
gris = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
gris_blur = cv2.medianBlur(gris, 7)

circulos = cv2.HoughCircles(
    gris_blur,
    cv2.HOUGH_GRADIENT,
    dp=1,
    minDist=100,
    param1=100,
    param2=50,
    minRadius=50,
    maxRadius=150,
)

print(f"Círculos detectados: {len(circulos[0])}")
for i, c in enumerate(circulos[0]):
    print(f"  Círculo {i}: centro=({c[0]:.0f}, {c[1]:.0f}), radio={c[2]:.1f} px")

# Paso 2: Calibración con la moneda más grande (2 euro = 25.75 mm)
radios = circulos[0][:, 2]
idx_mayor = np.argmax(radios)
radio_ref_px = radios[idx_mayor]

DIAMETRO_REF_MM = 25.75  # 2 euro
px_por_mm = (2 * radio_ref_px) / DIAMETRO_REF_MM

print(f"\nCalibración:")
print(f"  Moneda de referencia: 2 euro (diámetro = {DIAMETRO_REF_MM} mm)")
print(f"  Radio en píxeles: {radio_ref_px:.1f}")
print(f"  Relación: {px_por_mm:.2f} px/mm")


### Calibración interactiva (opcional)

Si se desea calibrar manualmente haciendo clic en una moneda de 1€, descomentar y ejecutar
la celda siguiente. Se abrirá una ventana OpenCV donde se puede hacer clic en la moneda de 1€.


In [ ]:
# --- CALIBRACIÓN INTERACTIVA (OPCIONAL) ---
# Descomentar para calibrar haciendo clic en la moneda de 1 euro

# clicked_point = None
# 
# def on_click(event, x, y, flags, param):
#     global clicked_point
#     if event == cv2.EVENT_LBUTTONDOWN:
#         clicked_point = (x, y)
# 
# img_display = img.copy()
# cv2.putText(img_display, "Haz clic en la moneda de 1 euro", (10, 30),
#             cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
# cv2.namedWindow("Calibracion", cv2.WINDOW_NORMAL)
# cv2.setMouseCallback("Calibracion", on_click)
# cv2.imshow("Calibracion", img_display)
# cv2.waitKey(0)
# cv2.destroyAllWindows()
# 
# if clicked_point is not None:
#     min_dist_click = float("inf")
#     for c in circulos[0]:
#         d = np.sqrt((c[0] - clicked_point[0])**2 + (c[1] - clicked_point[1])**2)
#         if d < min_dist_click:
#             min_dist_click = d
#             radio_ref_px = c[2]
#     DIAMETRO_REF_MM = 23.25  # 1 euro
#     px_por_mm = (2 * radio_ref_px) / DIAMETRO_REF_MM
#     print(f"Calibración por clic: {px_por_mm:.2f} px/mm")

print(f"Usando calibración automática: {px_por_mm:.2f} px/mm")


In [ ]:
# Detección y clasificación de todas las monedas
img_resultado, monedas, total = detectar_y_contar_monedas(
    img, px_por_mm,
    min_radius_px=50, max_radius_px=150,
    param1=100, param2=50, min_dist=100,
    blur_ksize=7
)

# Mostrar resultados
plt.figure(figsize=(8, 16))
plt.imshow(cv2.cvtColor(img_resultado, cv2.COLOR_BGR2RGB))
plt.title(f"Monedas detectadas - Total: {total:.2f} EUR")
plt.axis("off")
plt.show()

# Resumen detallado
print("\n" + "="*60)
print("RESUMEN DE MONEDAS DETECTADAS")
print("="*60)
for i, m in enumerate(monedas):
    print(f'  {i+1}. {m["nombre"]:>15s} | {m["valor"]:>5.2f} EUR | '
          f'diam {m["diametro_mm"]:>5.1f} mm | color: {m["tipo_color"]}')
print("-"*60)
print(f"  TOTAL: {total:.2f} EUR  ({len(monedas)} monedas)")
print("="*60)


## Análisis de la detección en imagen estática

La imagen contiene las 8 monedas Euro estándar:
- **2€, 1€** (bimetálicas)
- **50c, 20c, 10c** (doradas/Nordic gold)
- **5c, 2c, 1c** (cobrizas)

El valor total esperado es: **3.88 €**

La clasificación combina:
1. **Tamaño (diámetro en mm)**: Obtenido de la Transformada de Hough + calibración
2. **Color (HSV)**: Distingue bimetálicas, doradas y cobrizas


## Paso 2: Demostrador en vivo (Webcam)

El siguiente código captura vídeo de la webcam y detecta monedas en tiempo real.

### Instrucciones:
1. Ejecuta la celda siguiente.
2. Se abrirá una ventana con la imagen de la webcam.
3. Calibra: pulsa `c` y haz clic en una moneda de 1€, o pulsa `a` (auto-calibración con la más grande = 2€).
4. La detección se realiza continuamente.
5. Pulsa `q` o `ESC` para salir.

### Controles:
- `c`: Modo calibración (clic en moneda de 1€)
- `a`: Calibración automática (moneda más grande = 2€)
- `+`/`-`: Ajustar sensibilidad del detector
- `q` o `ESC`: Salir


In [ ]:
def demo_webcam():
    """Demostrador en vivo de detección y conteo de monedas con webcam."""
    cap = cv2.VideoCapture(0)
    if not cap.isOpened():
        print("Error: No se pudo abrir la webcam.")
        return
    
    px_por_mm_live = None
    modo_calibracion = False
    clicked_point_live = None
    param2_hough = 50
    
    def on_mouse(event, x, y, flags, param):
        nonlocal clicked_point_live
        if event == cv2.EVENT_LBUTTONDOWN and modo_calibracion:
            clicked_point_live = (x, y)
    
    cv2.namedWindow("Detector de Monedas", cv2.WINDOW_NORMAL)
    cv2.setMouseCallback("Detector de Monedas", on_mouse)
    
    print("Controles:")
    print("  c - Modo calibración (clic en moneda de 1 euro)")
    print("  a - Calibración automática (moneda más grande = 2 euro)")
    print("  +/- - Ajustar sensibilidad")
    print("  q/ESC - Salir")
    
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        
        display_frame = frame.copy()
        gris = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        gris_blur = cv2.medianBlur(gris, 7)
        
        # Detectar círculos
        h_frame, w_frame = frame.shape[:2]
        min_r = max(15, int(min(h_frame, w_frame) * 0.03))
        max_r = int(min(h_frame, w_frame) * 0.25)
        min_d = max(30, int(min(h_frame, w_frame) * 0.08))
        
        circulos_live = cv2.HoughCircles(
            gris_blur,
            cv2.HOUGH_GRADIENT,
            dp=1,
            minDist=min_d,
            param1=100,
            param2=param2_hough,
            minRadius=min_r,
            maxRadius=max_r,
        )
        
        # Modo calibración
        if modo_calibracion:
            cv2.putText(display_frame, "CALIBRACION: Clic en moneda de 1 euro",
                       (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2)
            
            if clicked_point_live is not None and circulos_live is not None:
                best_dist = float("inf")
                best_radio = None
                for c in circulos_live[0]:
                    d = np.sqrt((c[0]-clicked_point_live[0])**2 + (c[1]-clicked_point_live[1])**2)
                    if d < best_dist:
                        best_dist = d
                        best_radio = c[2]
                if best_radio is not None:
                    px_por_mm_live = (2 * best_radio) / 23.25  # 1 euro
                    print(f"Calibrado: {px_por_mm_live:.2f} px/mm")
                modo_calibracion = False
                clicked_point_live = None
        
        # Si está calibrado, clasificar monedas
        if px_por_mm_live is not None and circulos_live is not None:
            total_live = 0.0
            n_monedas = 0
            for circ in circulos_live[0]:
                cx, cy, radio_px = circ
                diametro_mm = (2 * radio_px) / px_por_mm_live
                tipo_color = clasificar_color_moneda(frame, cx, cy, radio_px)
                nombre, valor = clasificar_moneda(diametro_mm, tipo_color)
                total_live += valor
                n_monedas += 1
                
                color_d = COLORES_TIPO.get(tipo_color, (255, 255, 255))
                cv2.circle(display_frame, (int(cx), int(cy)), int(radio_px), color_d, 2)
                cv2.circle(display_frame, (int(cx), int(cy)), 3, color_d, -1)
                cv2.putText(display_frame, f"{nombre} ({valor:.2f}E)",
                           (int(cx)-int(radio_px), int(cy)-int(radio_px)-10),
                           cv2.FONT_HERSHEY_SIMPLEX, 0.4, color_d, 1)
            
            cv2.putText(display_frame, f"Total: {total_live:.2f} EUR ({n_monedas} monedas)",
                       (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)
        elif px_por_mm_live is None:
            cv2.putText(display_frame, "Sin calibrar. Pulsa c o a",
                       (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2)
            if circulos_live is not None:
                for circ in circulos_live[0]:
                    cv2.circle(display_frame, (int(circ[0]), int(circ[1])),
                             int(circ[2]), (200, 200, 200), 1)
        
        cv2.putText(display_frame, f"Sensibilidad: {param2_hough} (+/-)",
                   (10, h_frame - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (200, 200, 200), 1)
        
        cv2.imshow("Detector de Monedas", display_frame)
        
        key = cv2.waitKey(30) & 0xFF
        if key == ord("q") or key == 27:
            break
        elif key == ord("c"):
            modo_calibracion = True
            clicked_point_live = None
            print("Modo calibración. Haz clic en moneda de 1 euro.")
        elif key == ord("a"):
            if circulos_live is not None:
                radios_live = circulos_live[0][:, 2]
                radio_max = np.max(radios_live)
                px_por_mm_live = (2 * radio_max) / 25.75
                print(f"Auto-calibrado (2 euro): {px_por_mm_live:.2f} px/mm")
        elif key == ord("+") or key == ord("="):
            param2_hough = min(100, param2_hough + 5)
            print(f"Sensibilidad: {param2_hough}")
        elif key == ord("-"):
            param2_hough = max(10, param2_hough - 5)
            print(f"Sensibilidad: {param2_hough}")
    
    cap.release()
    cv2.destroyAllWindows()
    print("Webcam cerrada.")

# Ejecutar el demostrador
demo_webcam()


## Observaciones y limitaciones

### Funcionamiento correcto
- La detección funciona bien con fondo uniforme y monedas separadas.
- La combinación de tamaño + color mejora significativamente la clasificación.
- La calibración permite adaptar el sistema a diferentes distancias de la cámara.

### Problemas observados
- **Iluminación variable**: Cambios de luz afectan a la detección Hough y a la clasificación por color.
- **Oclusión/Solape**: Monedas solapadas no se detectan correctamente con Hough.
- **Fondo complejo**: Bordes y texturas del fondo generan falsos círculos.
- **Monedas similares en tamaño**: 50c (24.25mm) vs 2€ (25.75mm), 20c (22.25mm) vs 1€ (23.25mm). El color es clave.
- **Perspectiva**: La cámara no perpendicular hace que las monedas aparezcan como elipses.

### Posibles mejoras
- Usar detección de elipses para manejar perspectiva.
- Ecualización del histograma para robustez ante cambios de iluminación.
- Umbralizado adaptativo + contornos como alternativa a Hough.
- Descriptores de textura (LBP, Gabor) para mejorar la distinción entre monedas.
